# Tarea: Construcción del Dataset Integrado (Demografía, Salud y Economía)

**Objetivo:** extender el dataset demográfico construido en `U3_2_Preparacion_dataset.ipynb`
agregando las variables `percentage_health` y `gdp`.

La salida final debe contener únicamente estas 8 columnas:
`entity, year, population, births, deaths, child_deaths_under_5, percentage_health, gdp`

In [1]:
import pandas as pd

OPCIONES_OWID = {"User-Agent": "Our World In Data data fetch/1.0"}

## 1. Dataset base (igual al de la práctica de preparación de datos)

In [2]:
url_pop = "https://ourworldindata.org/grapher/population-with-un-projections.csv?v=1&csvType=full&useColumnShortNames=true"
url_births = "https://ourworldindata.org/grapher/number-of-births-per-year.csv?v=1&csvType=full&useColumnShortNames=true"
url_deaths = "https://ourworldindata.org/grapher/number-of-deaths-per-year.csv?v=1&csvType=full&useColumnShortNames=true"
url_child = "https://ourworldindata.org/grapher/number-of-child-deaths-unwpp.csv?v=1&csvType=full&useColumnShortNames=true"

poblacion = pd.read_csv(url_pop, storage_options=OPCIONES_OWID)
nacimientos = pd.read_csv(url_births, storage_options=OPCIONES_OWID)
defunciones = pd.read_csv(url_deaths, storage_options=OPCIONES_OWID)
defunciones_menores = pd.read_csv(url_child, storage_options=OPCIONES_OWID)

col_pop = [c for c in poblacion.columns if "population" in c and "estimates" in c][0]
col_births = [c for c in nacimientos.columns if "births" in c and "estimates" in c][0]
col_deaths = [c for c in defunciones.columns if "deaths" in c and "estimates" in c][0]
col_child = [c for c in defunciones_menores.columns if c not in ("entity", "code", "year")][0]

poblacion = poblacion[["entity", "code", "year", col_pop]].rename(columns={col_pop: "population"})
nacimientos = nacimientos[["entity", "code", "year", col_births]].rename(columns={col_births: "births"})
defunciones = defunciones[["entity", "code", "year", col_deaths]].rename(columns={col_deaths: "deaths"})
defunciones_menores = defunciones_menores[["entity", "code", "year", col_child]].rename(
    columns={col_child: "child_deaths_under_5"}
)

base = (
    poblacion
    .merge(nacimientos, on=["entity", "code", "year"], how="inner")
    .merge(defunciones, on=["entity", "code", "year"], how="inner")
    .merge(defunciones_menores, on=["entity", "code", "year"], how="inner")
)

print("Dataset base:", base.shape)
base.head(3)

Dataset base: (18720, 7)


,entity,code,year,population,births,deaths,child_deaths_under_5
0,Afghanistan,AFG,1950,7776180.0,383985.0,290972.0,160791
1,Afghanistan,AFG,1951,7879343.0,391002.0,288752.0,158566
2,Afghanistan,AFG,1952,7987784.0,397663.0,288059.0,157832


## 2. Nuevos indicadores: gasto en salud (% PIB) y PIB per cápita

In [3]:
url_salud = "https://ourworldindata.org/grapher/total-healthcare-expenditure-gdp.csv?v=1&csvType=full&useColumnShortNames=true"
url_pib = "https://ourworldindata.org/grapher/gdp-per-capita-worldbank.csv?v=1&csvType=full&useColumnShortNames=true"

gasto_salud = pd.read_csv(url_salud, storage_options=OPCIONES_OWID)
pib = pd.read_csv(url_pib, storage_options=OPCIONES_OWID)

col_gasto = [c for c in gasto_salud.columns if "health_expenditure" in c][0]
col_pib = [c for c in pib.columns if c.startswith("ny_gdp_pcap")][0]

gasto_salud = gasto_salud[["entity", "year", col_gasto]].rename(
    columns={col_gasto: "percentage_health"}
)
pib = pib[["entity", "year", col_pib]].rename(columns={col_pib: "gdp"})

gasto_salud.head(3)

,entity,year,percentage_health
0,Afghanistan,2002,9.443391
1,Afghanistan,2003,8.941258
2,Afghanistan,2004,9.808474


## 3. Dataset final (8 columnas)

In [4]:
dataset_final = (
    base
    .merge(gasto_salud, on=["entity", "year"], how="inner")
    .merge(pib, on=["entity", "year"], how="inner")
)

columnas_finales = [
    "entity", "year", "population", "births", "deaths",
    "child_deaths_under_5", "percentage_health", "gdp",
]
dataset_final = dataset_final[columnas_finales].sort_values(["entity", "year"]).reset_index(drop=True)

print("Dimensiones finales:", dataset_final.shape)
dataset_final.head()

Dimensiones finales: (4428, 8)


,entity,year,population,births,deaths,child_deaths_under_5,percentage_health,gdp
0,Afghanistan,2002,21378123.0,1015202.0,239161.0,123215,9.443391,1774.3087
1,Afghanistan,2003,22733053.0,1069727.0,243573.0,123894,8.941258,1815.9282
2,Afghanistan,2004,23560656.0,1095372.0,244435.0,122315,9.808474,1776.9182
3,Afghanistan,2005,24404575.0,1097038.0,241932.0,118420,9.948289,1908.1147
4,Afghanistan,2006,25424100.0,1120880.0,245586.0,116622,10.622766,1929.7239


In [5]:
dataset_final.info()

<class 'pandas.DataFrame'>
RangeIndex: 4428 entries, 0 to 4427
Data columns (total 8 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   entity                4428 non-null   str    
 1   year                  4428 non-null   int64  
 2   population            4428 non-null   float64
 3   births                4428 non-null   float64
 4   deaths                4428 non-null   float64
 5   child_deaths_under_5  4428 non-null   int64  
 6   percentage_health     4428 non-null   float64
 7   gdp                   4428 non-null   float64
dtypes: float64(5), int64(2), str(1)
memory usage: 276.9 KB


## 4. Revisión: México

In [6]:
dataset_final[dataset_final["entity"] == "Mexico"]

,entity,year,population,births,deaths,child_deaths_under_5,percentage_health,gdp
2545,Mexico,2000,98625553.0,2385977.0,540567.0,66928,4.244365,20470.494
2546,Mexico,2001,100099101.0,2380261.0,543803.0,64012,4.582014,20078.219
2547,Mexico,2002,101548627.0,2374140.0,553391.0,61210,4.830428,19744.793
2548,Mexico,2003,102978517.0,2351177.0,563140.0,58280,5.539788,19701.463
2549,Mexico,2004,104394131.0,2335226.0,575925.0,55625,5.683502,20127.225
2550,Mexico,2005,105811504.0,2318498.0,585859.0,53204,5.581101,20277.256
2551,Mexico,2006,107253666.0,2303997.0,597892.0,50857,5.406479,20965.826
2552,Mexico,2007,108774356.0,2296268.0,611318.0,48847,5.506680,21102.270
2553,Mexico,2008,110374286.0,2297950.0,627177.0,47137,5.445874,20992.560
2554,Mexico,2009,111999720.0,2303532.0,642292.0,45610,5.847865,19385.543


## 5. Guardar el resultado

In [7]:
ruta_salida = "../data/dataset_health_gdp_demografia_final.csv"
dataset_final.to_csv(ruta_salida, index=False)
print("Archivo guardado:", ruta_salida)

Archivo guardado: ../data/dataset_health_gdp_demografia_final.csv
